# Churn Intelligence Platform — Exploratory Data Analysis

This notebook explores the generated customer and activity data before building the prediction model.

**Run `python main.py --generate` first** to create the CSVs in `data/`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

customers = pd.read_csv("../data/customers.csv")
activity = pd.read_csv("../data/monthly_activity.csv")

print(f"Customers: {len(customers)} rows")
print(f"Activity:  {len(activity)} rows")
customers.head()

## 1. Churn Distribution
How many customers churned vs stayed?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
churn_counts = customers["churned"].value_counts()
axes[0].bar(["Active", "Churned"], churn_counts.values, color=["steelblue", "salmon"])
axes[0].set_title("Customer Count by Churn Status")
axes[0].set_ylabel("Count")

# By plan type
churn_by_plan = customers.groupby("plan_type")["churned"].mean().sort_values()
axes[1].barh(churn_by_plan.index, churn_by_plan.values, color="coral")
axes[1].set_title("Churn Rate by Plan Type")
axes[1].set_xlabel("Churn Rate")

plt.tight_layout()
plt.show()

## 2. Age Distribution
Are certain age groups more likely to churn?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(data=customers, x="age", hue="churned", multiple="stack", bins=30, ax=ax)
ax.set_title("Age Distribution by Churn Status")
plt.show()

## 3. Engagement: Churners vs Active Customers
Compare average monthly logins, sessions, and satisfaction between the two groups.

In [ ]:
# Merge activity with churn labels
merged = activity.merge(customers[["customer_id", "churned"]], on="customer_id")

metrics = ["logins", "total_sessions", "feature_usage_score", "satisfaction_score"]
fig, axes = plt.subplots(1, len(metrics), figsize=(16, 4))

for ax, col in zip(axes, metrics):
    sns.boxplot(data=merged, x="churned", y=col, ax=ax, palette=["steelblue", "salmon"])
    ax.set_xticklabels(["Active", "Churned"])
    ax.set_title(col.replace("_", " ").title())

plt.tight_layout()
plt.show()

## 4. Support Tickets & Payment Failures
Do churners file more tickets and miss more payments?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Average tickets per month
ticket_avg = merged.groupby("churned")["support_tickets"].mean()
axes[0].bar(["Active", "Churned"], ticket_avg.values, color=["steelblue", "salmon"])
axes[0].set_title("Avg Support Tickets per Month")
axes[0].set_ylabel("Tickets")

# Payment status breakdown
pay_breakdown = merged.groupby(["churned", "payment_status"]).size().unstack(fill_value=0)
pay_pct = pay_breakdown.div(pay_breakdown.sum(axis=1), axis=0)
pay_pct.plot(kind="bar", stacked=True, ax=axes[1], color=["salmon", "gold", "steelblue"])
axes[1].set_xticklabels(["Active", "Churned"], rotation=0)
axes[1].set_title("Payment Status Breakdown")
axes[1].set_ylabel("Proportion")
axes[1].legend(title="Status")

plt.tight_layout()
plt.show()

## 5. Monthly Engagement Trend
How does average engagement change month-over-month for churners vs active customers?

In [ ]:
monthly_trend = merged.groupby(["month", "churned"])["logins"].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
for label, grp in monthly_trend.groupby("churned"):
    name = "Churned" if label else "Active"
    ax.plot(grp["month"], grp["logins"], label=name, marker="o", markersize=3)

ax.set_title("Average Monthly Logins: Active vs Churned")
ax.set_xlabel("Month")
ax.set_ylabel("Avg Logins")
ax.legend()
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap
Which numeric features correlate with each other and with churn?

In [ ]:
numeric_cols = ["logins", "total_sessions", "feature_usage_score",
                "avg_session_duration", "support_tickets", "satisfaction_score"]
merged["churned_int"] = merged["churned"].astype(int)
corr = merged[numeric_cols + ["churned_int"]].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()